# Batch Normalization – Lesson

1. [What is it?](#theory)
2. [What are it's benefits?](#benefits)
3. [How do we add it to a network?](#implementation_1)
4. [Let's see it work!](#demos)
5. [What are you hiding?](#implementation_2)

# What is Batch Normalization?<a id='theory'></a>

Batch normalization was introduced in Sergey Ioffe's and Christian Szegedy's 2015 paper [Batch Normalization: Accelerating Deep Network Training by Reducing Internal Covariate Shift](https://arxiv.org/pdf/1502.03167.pdf). The idea is that, instead of just normalizing the inputs to the network, we normalize the inputs to _layers within_ the network. It's called "batch" normalization because during training, we normalize each layer's inputs by using the mean and variance of the values in the current mini-batch.

Why might this help? Well, we know that normalizing the inputs to a _network_ helps the network learn. But a network is a series of layers, where the output of one layer becomes the input to another. That means we can think of any layer in a neural network as the _first_ layer of a smaller network.

For example, imagine a 3 layer network. Instead of just thinking of it as a single network with inputs, layers, and outputs, think of the output of layer 1 as the input to a two layer network. This two layer network would consist of layers 2 and 3 in our original network. 

Likewise, the output of layer 2 can be thought of as the input to a single layer network, consisting only of layer 3.

When you think of it like that - as a series of neural networks feeding into each other - then it's easy to imagine how normalizing the inputs to each layer would help. It's just like normalizing the inputs to any other neural network, but you're doing it at every layer (sub-network).

Beyond the intuitive reasons, there are good mathematical reasons why it helps the network learn better, too. It helps combat what the authors call _internal covariate shift_. This discussion is best handled [in the paper](https://arxiv.org/pdf/1502.03167.pdf) and in [Deep Learning](http://www.deeplearningbook.org) a book you can read online written by Ian Goodfellow, Yoshua Bengio, and Aaron Courville. Specifically, check out the batch normalization section of [Chapter 8: Optimization for Training Deep Models](http://www.deeplearningbook.org/contents/optimization.html).

# Benefits of Batch Normalization<a id="benefits"></a>

Batch normalization optimizes network training. It has been shown to have several benefits:
1. **Networks train faster** – Each training _iteration_ will actually be slower because of the extra calculations during the forward pass and the additional hyperparameters to train during back propagation. However, it should converge much more quickly, so training should be faster overall. 
2. **Allows higher learning rates** – Gradient descent usually requires small learning rates for the network to converge. And as networks get deeper, their gradients get smaller during back propagation so they require even more iterations. Using batch normalization allows us to use much higher learning rates, which further increases the speed at which networks train. 
3. **Makes weights easier to initialize** – Weight initialization can be difficult, and it's even more difficult when creating deeper networks. Batch normalization seems to allow us to be much less careful about choosing our initial starting weights.  
4. **Makes more activation functions viable** – Some activation functions do not work well in some situations. Sigmoids lose their gradient pretty quickly, which means they can't be used in deep networks. And ReLUs often die out during training, where they stop learning completely, so we need to be careful about the range of values fed into them. Because batch normalization regulates the values going into each activation function, non-linearlities that don't seem to work well in deep networks actually become viable again.  
5. **Simplifies the creation of deeper networks** – Because of the first 4 items listed above, it is easier to build and faster to train deeper neural networks when using batch normalization. And it's been shown that deeper networks generally produce better results, so that's great.
6. **Provides a bit of regularlization** – Batch normalization adds a little noise to your network. In some cases, such as in Inception modules, batch normalization has been shown to work as well as dropout. But in general, consider batch normalization as a bit of extra regularization, possibly allowing you to reduce some of the dropout you might add to a network. 
7. **May give better results overall** – Some tests seem to show batch normalization actually improves the training results. However, it's really an optimization to help train faster, so you shouldn't think of it as a way to make your network better. But since it lets you train networks faster, that means you can iterate over more designs more quickly. It also lets you build deeper networks, which are usually better. So when you factor in everything, you're probably going to end up with better results if you build your networks with batch normalization.

# Batch Normalization in PyTorch<a id="implementation_1"></a>

This section of the notebook shows you one way to add batch normalization to a neural network built in PyTorch.

The following cell imports the packages we need in the notebook and loads the MNIST dataset to use in our experiments. However, the `torch.nn` package contains all the code you'll actually need for batch normalization.

In [ ]:
# Import necessary packages
import os

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
import tqdm
from torch import nn
%matplotlib inline

import helper

# Use the GPU when there is one; otherwise fall back to the CPU.
device = helper.device
print("Using device:", device)

# Set the environment variable DL_SMOKE=1 to train for far fewer batches (a quick check that everything runs).
SMOKE = os.environ.get('DL_SMOKE') == '1'

# Import MNIST data so we have something for our experiments
mnist = helper.load_mnist()

### Neural network classes for testing

The following class, `NeuralNet`, allows us to create identical neural networks with and without batch normalization. The code is heavily documented, but there is also some additional discussion later. You do not need to read through it all before going through the rest of the notebook, but the comments within the code blocks may answer some of your questions.

*About the code:*
>This class is not meant to represent PyTorch best practices – the design choices made here are to support the discussion related to batch normalization.

>It's also important to note that we use the well-known MNIST data for these examples, but the networks we create are not meant to be good for performing handwritten character recognition. We chose this network architecture because it is similar to the one used in the original paper, which is complex enough to demonstrate some of the benefits of batch normalization while still being fast to train.

In [ ]:
class NeuralNet:
    def __init__(self, initial_weights, activation_fn, use_batch_norm, batch_norm_layer=nn.BatchNorm1d):
        """
        Initializes this object, creating a PyTorch network using the given parameters.

        :param initial_weights: list of NumPy arrays or Tensors
            Initial values for the weights for every layer in the network. We pass these in
            so we can create multiple networks with the same starting weights to eliminate
            training differences caused by random initialization differences.
            The number of items in the list defines the number of layers in the network,
            and the shapes of the items in the list define the number of nodes in each layer.
            e.g. Passing in 3 matrices of shape (784, 256), (256, 100), and (100, 10) would
            create a network with 784 inputs going into a hidden layer with 256 nodes,
            followed by a hidden layer with 100 nodes, followed by an output layer with 10 nodes.
        :param activation_fn: Callable
            The function used for the output of each hidden layer. The network will use the same
            activation function on every hidden layer and no activation function on the output layer.
            e.g. Pass torch.relu to use ReLU activations on your hidden layers.
        :param use_batch_norm: bool
            Pass True to create a network that uses batch normalization; False otherwise
            Note: this network will not use batch normalization on layers that do not have an
            activation function.
        :param batch_norm_layer: Callable (default nn.BatchNorm1d)
            The layer class that does the batch normalization. It is called as
            `batch_norm_layer(num_features, eps=..., momentum=...)`. Later in this notebook we pass in our
            own implementation instead of PyTorch's.
        """
        # Keep track of whether or not this network uses batch normalization.
        self.use_batch_norm = use_batch_norm
        self.name = "With Batch Norm" if use_batch_norm else "Without Batch Norm"

        # This list is just for keeping track of data we want to plot later.
        # It doesn't actually have anything to do with neural nets or batch normalization.
        self.training_accuracies = []

        # Create the network. PyTorch layers know whether they are in training or inference mode
        # (see `network.train()` and `network.eval()` below), which batch normalization needs because
        # it does different calculations in each.
        self.network = self.build_network(initial_weights, activation_fn, batch_norm_layer).to(device)

    def build_network(self, initial_weights, activation_fn, batch_norm_layer):
        """
        Build the network. It still needs to be trained via the `train` method.

        :param initial_weights: list of NumPy arrays or Tensors
            See __init__ for description.
        :param activation_fn: Callable
            See __init__ for description.
        """
        layers = []
        for weights in initial_weights[:-1]:
            layers += self.fully_connected(weights, activation_fn, batch_norm_layer)
        layers += self.fully_connected(initial_weights[-1])
        return nn.Sequential(*layers)

    def fully_connected(self, initial_weights, activation_fn=None, batch_norm_layer=nn.BatchNorm1d):
        """
        Creates a standard, fully connected layer. Its number of inputs and outputs will be
        defined by the shape of `initial_weights`, and its starting weight values will be
        taken directly from that same parameter. If `self.use_batch_norm` is True, this
        layer will include batch normalization, otherwise it will not.

        :param initial_weights: NumPy array or Tensor
            Initial values for this layer's weights. The shape defines the number of nodes in the layer.
            e.g. Passing in 3 matrix of shape (784, 256) would create a layer with 784 inputs and 256
            outputs.
        :param activation_fn: Callable or None (default None)
            The non-linearity used for the output of the layer. If None, this layer will not include
            batch normalization, regardless of the value of `self.use_batch_norm`.
            e.g. Pass torch.relu to use ReLU activations on your hidden layers.
        :returns: a list of modules that make up the layer
        """
        num_in, num_out = initial_weights.shape

        # Since this class supports both options, only use batch normalization when
        # requested. However, do not use it on the final layer, which we identify
        # by its lack of an activation function.
        use_batch_norm = self.use_batch_norm and activation_fn is not None

        # Batch normalization uses weights as usual, but does NOT add a bias term. This is because
        # its calculations include gamma and beta variables that make the bias term unnecessary.
        # (See later in the notebook for more details.)
        linear = nn.Linear(num_in, num_out, bias=not use_batch_norm)
        with torch.no_grad():
            # nn.Linear stores its weights as (outputs, inputs), so we transpose ours
            linear.weight.copy_(torch.as_tensor(initial_weights).T)
            if linear.bias is not None:
                linear.bias.zero_()
        layers = [linear]

        if use_batch_norm:
            # Apply batch normalization to the linear combination of the inputs and weights.
            # (eps=1e-3 and momentum=0.01 are the values the original TensorFlow version of this lesson used.
            #  PyTorch's momentum is the weight given to the *new* batch statistics.)
            layers.append(batch_norm_layer(num_out, eps=1e-3, momentum=0.01))

        if activation_fn is not None:
            # Now apply the activation function, *after* the normalization.
            layers.append(Activation(activation_fn))
        return layers

    def accuracy(self, images, labels, training_mode=False):
        """Fraction of `images` classified correctly, with the network in training or inference mode."""
        self.network.train(training_mode)
        with torch.no_grad():
            return (self.network(images).argmax(dim=1) == labels).float().mean().item()

    def train(self, learning_rate, training_batches, batches_per_sample):
        """
        Trains the model on the MNIST training dataset.

        :param learning_rate: float
            Learning rate used during gradient descent.
        :param training_batches: int
            Number of batches to train.
        :param batches_per_sample: int
            How many batches to train before sampling the validation accuracy.
        """
        optimizer = torch.optim.SGD(self.network.parameters(), lr=learning_rate)

        # Train for the appropriate number of batches. (tqdm is only for a nice timing display)
        for i in tqdm.tqdm(range(training_batches)):
            # We use batches of 60 just because the original paper did. You can use any size batch you like.
            batch_xs, batch_ys = mnist.train.next_batch(60)

            # Training mode: batch normalization uses the batch's mean and variance, and updates its
            # estimates of the population mean and variance.
            self.network.train()
            optimizer.zero_grad()
            loss = F.cross_entropy(self.network(batch_xs), batch_ys)
            loss.backward()
            optimizer.step()

            # Periodically test accuracy against the 5k validation images and store it for plotting later.
            if i % batches_per_sample == 0:
                self.training_accuracies.append(self.accuracy(mnist.validation.images, mnist.validation.labels))

        # After training, report accuracy against validation data
        validation_accuracy = self.accuracy(mnist.validation.images, mnist.validation.labels)
        print('{}: After training, final accuracy on validation set = {}'.format(self.name, validation_accuracy))

    def test(self, test_training_accuracy=False, include_individual_predictions=False):
        """
        Tests a trained model on the MNIST testing dataset.

        :param test_training_accuracy: bool (default False)
            If True, perform inference with batch normalization using batch mean and variance;
            if False, perform inference with batch normalization using estimated population mean and variance.
            Note: in real life, *always* perform inference using the population mean and variance.
                  This parameter exists just to support demonstrating what happens if you don't.
        :param include_individual_predictions: bool (default False)
            This function always performs an accuracy test against the entire test set. But if this parameter
            is True, it performs an extra test, doing 200 predictions one at a time, and displays the results
            and accuracy.
        """
        # Test against all of the MNIST test data
        test_accuracy = self.accuracy(mnist.test.images, mnist.test.labels, training_mode=test_training_accuracy)
        print('-'*75)
        print('{}: Accuracy on full test set = {}'.format(self.name, test_accuracy))

        # If requested, perform tests predicting individual values rather than batches
        if include_individual_predictions:
            predictions = []
            correct = 0

            # Do 200 predictions, 1 at a time
            for i in range(200):
                # This is a normal prediction using an individual test case. However, notice
                # we put the network in training mode if `test_training_accuracy` is True.
                # Remember that will tell it whether it should use the batch mean & variance or
                # the population estimates that were calucated while training the model.
                image, label = mnist.test.images[i:i+1], mnist.test.labels[i:i+1]
                correct += self.accuracy(image, label, training_mode=test_training_accuracy)
                with torch.no_grad():
                    predictions.append(self.network(image).argmax(dim=1).item())

            print("200 Predictions:", predictions)
            print("Accuracy on 200 samples:", correct/200)


class Activation(nn.Module):
    """Wraps a function such as torch.relu or torch.sigmoid as a layer."""

    def __init__(self, fn):
        super().__init__()
        self.fn = fn

    def forward(self, x):
        return self.fn(x)

There are quite a few comments in the code, so those should answer most of your questions. However, let's take a look at the most important lines.

We add batch normalization to layers inside the `fully_connected` method. Here are some important points about that code:
1. Layers with batch normalization do not include a bias term.
2. We use PyTorch's [`nn.BatchNorm1d`](https://pytorch.org/docs/stable/generated/torch.nn.BatchNorm1d.html) layer to handle the math. (We show a hand-written version [later in the notebook](#implementation_2).)
3. A batch normalization layer behaves differently in training and inference mode. We switch modes with `self.network.train()` and `self.network.eval()` (or `self.network.train(False)`). This is an important step we'll talk about later.
4. We add the normalization **before** calling the activation function.

The batch norm layer also keeps running estimates of the population mean and variance, and PyTorch updates them for us every time we run a batch through the network in training mode, like in the following lines from `train`:
```python
self.network.train()
optimizer.zero_grad()
loss = F.cross_entropy(self.network(batch_xs), batch_ys)
```
(In TensorFlow 1 you had to remember to run extra update operations. PyTorch does it inside the layer.)

We'll go into more details later, but next we want to show some experiments that use this code and test networks with and without batch normalization.

# Batch Normalization Demos<a id='demos'></a>
This section of the notebook trains various networks with and without batch normalization to demonstrate some of the benefits mentioned earlier.

We'd like to thank the author of this blog post [Implementing Batch Normalization in Tensorflow](http://r2rt.com/implementing-batch-normalization-in-tensorflow.html). That post provided the idea of - and some of the code for - plotting the differences in accuracy during training, along with the idea for comparing multiple networks using the same initial weights.

**Note:** the numbers quoted in the commentary below come from runs of the original TensorFlow version of this lesson. Your numbers will differ a little from run to run (weights and batches are random), but the overall picture should be the same. Training the networks for tens of thousands of batches takes a while; set the environment variable `DL_SMOKE=1` before starting Jupyter if you only want a quick, much shorter version.

## Code to support testing

The following two functions support the demos we run in the notebook. 

The first function, `plot_training_accuracies`, simply plots the values found in the `training_accuracies` lists of the `NeuralNet` objects passed to it. If you look at the `train` function in `NeuralNet`, you'll see it that while it's training the network, it periodically measures validation accuracy and stores the results in that list. It does that just to support these plots.

The second function, `train_and_test`, creates two neural nets - one with and one without batch normalization. It then trains them both and tests them, calling `plot_training_accuracies` to plot how their accuracies changed over the course of training. The really imporant thing about this function is that it initializes the starting weights for the networks _outside_ of the networks and then passes them in. This lets it train both networks from the exact same starting weights, which eliminates performance differences that might result from (un)lucky initial weights.

In [ ]:
def plot_training_accuracies(*args, **kwargs):
    """
    Displays a plot of the accuracies calculated during training to demonstrate
    how many iterations it took for the model(s) to converge.

    :param args: One or more NeuralNet objects
        You can supply any number of NeuralNet objects as unnamed arguments
        and this will display their training accuracies. Be sure to call `train`
        the NeuralNets before calling this function.
    :param kwargs:
        You can supply any named parameters here, but `batches_per_sample` is the only
        one we look for. It should match the `batches_per_sample` value you passed
        to the `train` function.
    """
    fig, ax = plt.subplots()

    batches_per_sample = kwargs['batches_per_sample']

    for net in args:
        ax.plot(range(0,len(net.training_accuracies)*batches_per_sample,batches_per_sample),
                net.training_accuracies, label=net.name)
    ax.set_xlabel('Training steps')
    ax.set_ylabel('Accuracy')
    ax.set_title('Validation Accuracy During Training')
    ax.legend(loc=4)
    ax.set_ylim([0,1])
    plt.yticks(np.arange(0, 1.1, 0.1))
    plt.grid(True)
    plt.show()

def train_and_test(use_bad_weights, learning_rate, activation_fn, training_batches=50000, batches_per_sample=500):
    """
    Creates two networks, one with and one without batch normalization, then trains them
    with identical starting weights, layers, batches, etc. Finally tests and plots their accuracies.

    :param use_bad_weights: bool
        If True, initialize the weights of both networks to wildly inappropriate weights;
        if False, use reasonable starting weights.
    :param learning_rate: float
        Learning rate used during gradient descent.
    :param activation_fn: Callable
        The function used for the output of each hidden layer. The network will use the same
        activation function on every hidden layer and no activate function on the output layer.
        e.g. Pass torch.relu to use ReLU activations on your hidden layers.
    :param training_batches: (default 50000)
        Number of batches to train.
    :param batches_per_sample: (default 500)
        How many batches to train before sampling the validation accuracy.
    """
    if SMOKE:
        # Quick check mode: far fewer batches
        training_batches = min(training_batches, 1500)
        batches_per_sample = min(batches_per_sample, 50)

    # Use identical starting weights for each network to eliminate differences in
    # weight initialization as a cause for differences seen in training performance
    #
    # Note: The networks will use these weights to define the number of and shapes of
    #       its layers. The original batch normalization paper used 3 hidden layers
    #       with 100 nodes in each, followed by a 10 node output layer. These values
    #       build such a network, but feel free to experiment with different choices.
    #       However, the input size should always be 784 and the final output should be 10.
    if use_bad_weights:
        # These weights should be horrible because they have such a large standard deviation
        weights = [np.random.normal(size=(784,100), scale=5.0).astype(np.float32),
                   np.random.normal(size=(100,100), scale=5.0).astype(np.float32),
                   np.random.normal(size=(100,100), scale=5.0).astype(np.float32),
                   np.random.normal(size=(100,10), scale=5.0).astype(np.float32)
                  ]
    else:
        # These weights should be good because they have such a small standard deviation
        weights = [np.random.normal(size=(784,100), scale=0.05).astype(np.float32),
                   np.random.normal(size=(100,100), scale=0.05).astype(np.float32),
                   np.random.normal(size=(100,100), scale=0.05).astype(np.float32),
                   np.random.normal(size=(100,10), scale=0.05).astype(np.float32)
                  ]

    # build two versions of same network, 1 without and 1 with batch normalization
    no_bn = NeuralNet(weights, activation_fn, False)
    bn = NeuralNet(weights, activation_fn, True)

    # train and test the two models
    no_bn.train(learning_rate, training_batches, batches_per_sample)
    bn.train(learning_rate, training_batches, batches_per_sample)

    no_bn.test()
    bn.test()

    # Display a graph of how validation accuracies changed during training
    # so we can compare how the models trained and when they converged
    plot_training_accuracies(no_bn, bn, batches_per_sample=batches_per_sample)

## Comparisons between identical networks, with and without batch normalization

The next series of cells train networks with various settings to show the differences with and without batch normalization. They are meant to clearly demonstrate the effects of batch normalization. We include a deeper discussion of batch normalization later in the notebook.

**The following creates two networks using a ReLU activation function, a learning rate of 0.01, and reasonable starting weights.**

In [ ]:
train_and_test(False, 0.01, torch.relu)

As expected, both networks train well and eventually reach similar test accuracies. However, notice that the model with batch normalization converges slightly faster than the other network, reaching accuracies over 90% almost immediately and nearing its max acuracy in 10 or 15 thousand iterations. The other network takes about 3 thousand iterations to reach 90% and doesn't near its best accuracy until 30 thousand or more iterations.

If you look at the raw speed, you can see that without batch normalization we were computing over 1100 batches per second, whereas with batch normalization that goes down to just over 500. However, batch normalization allows us to perform fewer iterations and converge in less time over all. (We only trained for 50 thousand batches here so we could plot the comparison.)

**The following creates two networks with the same hyperparameters used in the previous example, but only trains for 2000 iterations.**

In [ ]:
train_and_test(False, 0.01, torch.relu, 2000, 50)

As you can see, using batch normalization produces a model with over 95% accuracy in only 2000 batches, and it was above 90% at somewhere around 500 batches. Without batch normalization, the model takes 1750 iterations just to hit 80% – the network with batch normalization hits that mark after around 200 iterations! (Note: if you run the code yourself, you'll see slightly different results each time because the starting weights - while the same for each model - are different for each run.)

In the above example, you should also notice that the networks trained fewer batches per second then what you saw in the previous example. That's because much of the time we're tracking is actually spent periodically performing inference to collect data for the plots. In this example we perform that inference every 50 batches instead of every 500, so generating the plot for this example requires 10 times the overhead for the same 2000 iterations.

**The following creates two networks using a sigmoid activation function, a learning rate of 0.01, and reasonable starting weights.**

In [ ]:
train_and_test(False, 0.01, torch.sigmoid)

With the number of layers we're using and this small learning rate, using a sigmoid activation function takes a long time to start learning. It eventually starts making progress, but it took over 45 thousand batches just to get over 80% accuracy. Using batch normalization gets to 90% in around one thousand batches. 

**The following creates two networks using a ReLU activation function, a learning rate of 1, and reasonable starting weights.**

In [ ]:
train_and_test(False, 1, torch.relu)

Now we're using ReLUs again, but with a larger learning rate. The plot shows how training started out pretty normally, with the network with batch normalization starting out faster than the other. But the higher learning rate bounces the accuracy around a bit more, and at some point the accuracy in the network without batch normalization just completely crashes. It's likely that too many ReLUs died off at this point because of the high learning rate.

The next cell shows the same test again. The network with batch normalization performs the same way, and the other suffers from the same problem again, but it manages to train longer before it happens.

In [ ]:
train_and_test(False, 1, torch.relu)

In both of the previous examples, the network with batch normalization manages to gets over 98% accuracy, and get near that result almost immediately. The higher learning rate allows the network to train extremely fast.

**The following creates two networks using a sigmoid activation function, a learning rate of 1, and reasonable starting weights.**

In [ ]:
train_and_test(False, 1, torch.sigmoid)

In this example, we switched to a sigmoid activation function. It appears to handle the higher learning rate well, with both networks achieving high accuracy.

The cell below shows a similar pair of networks trained for only 2000 iterations.

In [ ]:
train_and_test(False, 1, torch.sigmoid, 2000, 50)

As you can see, even though these parameters work well for both networks, the one with batch normalization gets over 90% in 400 or so batches, whereas the other takes over 1700. When training larger networks, these sorts of differences become more pronounced.

**The following creates two networks using a ReLU activation function, a learning rate of 2, and reasonable starting weights.**

In [ ]:
train_and_test(False, 2, torch.relu)

With this very large learning rate, the network with batch normalization trains fine and almost immediately manages 98% accuracy. However, the network without normalization doesn't learn at all.

**The following creates two networks using a sigmoid activation function, a learning rate of 2, and reasonable starting weights.**

In [ ]:
train_and_test(False, 2, torch.sigmoid)

Once again, using a sigmoid activation function with the larger learning rate works well both with and without batch normalization.

However, look at the plot below where we train models with the same parameters but only 2000 iterations. As usual, batch normalization lets it train faster. 

In [ ]:
train_and_test(False, 2, torch.sigmoid, 2000, 50)

In the rest of the examples, we use really bad starting weights. That is, normally we would use very small values close to zero. However, in these examples we choose random values with a standard deviation of 5. If you were really training a neural network, you would **not** want to do this. But these examples demonstrate how batch normalization makes your network much more resilient.  

**The following creates two networks using a ReLU activation function, a learning rate of 0.01, and bad starting weights.**

In [ ]:
train_and_test(True, 0.01, torch.relu)

As the plot shows, without batch normalization the network never learns anything at all. But with batch normalization, it actually learns pretty well and gets to almost 80% accuracy. The starting weights obviously hurt the network, but you can see how well batch normalization does in overcoming them. 

**The following creates two networks using a sigmoid activation function, a learning rate of 0.01, and bad starting weights.**

In [ ]:
train_and_test(True, 0.01, torch.sigmoid)

Using a sigmoid activation function works better than the ReLU in the previous example, but without batch normalization it would take a tremendously long time to train the network, if it ever trained at all. 

**The following creates two networks using a ReLU activation function, a learning rate of 1, and bad starting weights.**<a id="successful_example_lr_1"></a>

In [ ]:
train_and_test(True, 1, torch.relu)

The higher learning rate used here allows the network with batch normalization to surpass 90% in about 30 thousand batches. The network without it never gets anywhere.

**The following creates two networks using a sigmoid activation function, a learning rate of 1, and bad starting weights.**

In [ ]:
train_and_test(True, 1, torch.sigmoid)

Using sigmoid works better than ReLUs for this higher learning rate. However, you can see that without batch normalization, the network takes a long time tro train, bounces around a lot, and spends a long time stuck at 90%. The network with batch normalization trains much more quickly, seems to be more stable, and achieves a higher accuracy.

**The following creates two networks using a ReLU activation function, a learning rate of 2, and bad starting weights.**<a id="successful_example_lr_2"></a>

In [ ]:
train_and_test(True, 2, torch.relu)

We've already seen that ReLUs do not do as well as sigmoids with higher learning rates, and here we are using an extremely high rate. As expected, without batch normalization the network doesn't learn at all. But with batch normalization, it eventually achieves 90% accuracy. Notice, though, how its accuracy bounces around wildly during training - that's because the learning rate is really much too high, so the fact that this worked at all is a bit of luck.

**The following creates two networks using a sigmoid activation function, a learning rate of 2, and bad starting weights.**

In [ ]:
train_and_test(True, 2, torch.sigmoid)

In this case, the network with batch normalization trained faster and reached a higher accuracy. Meanwhile, the high learning rate makes the network without normalization bounce around erratically and have trouble getting past 90%.

### Full Disclosure: Batch Normalization Doesn't Fix Everything

Batch normalization isn't magic and it doesn't work every time. Weights are still randomly initialized and batches are chosen at random during training, so you never know exactly how training will go. Even for these tests, where we use the same initial weights for both networks, we still get _different_ weights each time we run.

This section includes two examples that show runs when batch normalization did not help at all.

**The following creates two networks using a ReLU activation function, a learning rate of 1, and bad starting weights.**

In [ ]:
train_and_test(True, 1, torch.relu)

When we used these same parameters [earlier](#successful_example_lr_1), we saw the network with batch normalization reach 92% validation accuracy. This time we used different starting weights, initialized using the same standard deviation as before, and the network doesn't learn at all. (Remember, an accuracy around 10% is what the network gets if it just guesses the same value all the time.)

**The following creates two networks using a ReLU activation function, a learning rate of 2, and bad starting weights.**

In [ ]:
train_and_test(True, 2, torch.relu)

When we trained with these parameters and batch normalization [earlier](#successful_example_lr_2), we reached 90% validation accuracy. However, this time the network _almost_ starts to make some progress in the beginning, but it quickly breaks down and stops learning. 

**Note:** Both of the above examples use *extremely* bad starting weights, along with learning rates that are too high. While we've shown batch normalization _can_ overcome bad values, we don't mean to encourage actually using them. The examples in this notebook are meant to show that batch normalization can help your networks train better. But these last two examples should remind you that you still want to try to use good network design choices and reasonable starting weights. It should also remind you that the results of each attempt to train a network are a bit random, even when using otherwise identical architectures.

# Batch Normalization: A Detailed Look<a id='implementation_2'></a>

The layer created by `nn.BatchNorm1d` handles all the details of implementing batch normalization. Many students will be fine just using that and won't care about what's happening at the lower levels. However, some students may want to explore the details, so here is a short explanation of what's really happening, starting with the equations you're likely to come across if you ever read about batch normalization. 

In order to normalize the values, we first need to find the average value for the batch. If you look at the code, you can see that this is not the average value of the batch _inputs_, but the average value coming _out_ of any particular layer before we pass it through its non-linear activation function and then feed it as an input to the _next_ layer.

We represent the average as $\mu_B$, which is simply the sum of all of the values $x_i$ divided by the number of values, $m$ 

$$
\mu_B \leftarrow \frac{1}{m}\sum_{i=1}^m x_i
$$

We then need to calculate the variance, or mean squared deviation, represented as $\sigma_{B}^{2}$. If you aren't familiar with statistics, that simply means for each value $x_i$, we subtract the average value (calculated earlier as $\mu_B$), which gives us what's called the "deviation" for that value. We square the result to get the squared deviation. Sum up the results of doing that for each of the values, then divide by the number of values, again $m$, to get the average, or mean, squared deviation.

$$
\sigma_{B}^{2} \leftarrow \frac{1}{m}\sum_{i=1}^m (x_i - \mu_B)^2
$$

Once we have the mean and variance, we can use them to normalize the values with the following equation. For each value, it subtracts the mean and divides by the (almost) standard deviation. (You've probably heard of standard deviation many times, but if you have not studied statistics you might not know that the standard deviation is actually the square root of the mean squared deviation.)

$$
\hat{x_i} \leftarrow \frac{x_i - \mu_B}{\sqrt{\sigma_{B}^{2} + \epsilon}}
$$

Above, we said "(almost) standard deviation". That's because the real standard deviation for the batch is calculated by $\sqrt{\sigma_{B}^{2}}$, but the above formula adds the term epsilon, $\epsilon$, before taking the square root. The epsilon can be any small, positive constant - in our code we use the value `0.001`. It is there partially to make sure we don't try to divide by zero, but it also acts to increase the variance slightly for each batch. 

Why increase the variance? Statistically, this makes sense because even though we are normalizing one batch at a time, we are also trying to estimate the population distribution – the total training set, which itself an estimate of the larger population of inputs your network wants to handle. The variance of a population is higher than the variance for any sample taken from that population, so increasing the variance a little bit for each batch helps take that into account. 

At this point, we have a normalized value, represented as $\hat{x_i}$. But rather than use it directly, we multiply it by a gamma value, $\gamma$, and then add a beta value, $\beta$. Both $\gamma$ and $\beta$ are learnable parameters of the network and serve to scale and shift the normalized value, respectively. Because they are learnable just like weights, they give your network some extra knobs to tweak during training to help it learn the function it is trying to approximate.  

$$
y_i \leftarrow \gamma \hat{x_i} + \beta
$$

We now have the final batch-normalized output of our layer, which we would then pass to a non-linear activation function like sigmoid, tanh, ReLU, Leaky ReLU, etc. In the original batch normalization paper (linked in the beginning of this notebook), they mention that there might be cases when you'd want to perform the batch normalization _after_ the non-linearity instead of before, but it is difficult to find any uses like that in practice.

In `NeuralNet`'s implementation of `fully_connected`, all of this math is hidden inside the following layer, which receives the linear combination of the inputs and weights (the $x_i$ from the equations) as its input:
```python
layers.append(batch_norm_layer(num_out, eps=1e-3, momentum=0.01))   # nn.BatchNorm1d by default
```
The next section shows you how to implement the math directly. 

### Batch normalization without the `nn.BatchNorm1d` layer

Our implementation of batch normalization in `NeuralNet` uses PyTorch's ready-made [`nn.BatchNorm1d`](https://pytorch.org/docs/stable/generated/torch.nn.BatchNorm1d.html) layer.

However, if you would like to implement batch normalization at a lower level, the following code shows you how. It writes the layer by hand using only basic tensor operations (`mean`, `var`, arithmetic).

**1)** You can pass this class to `NeuralNet` as its `batch_norm_layer` and everything in `NeuralNet` will still work like it did before, for example `NeuralNet(weights, torch.relu, True, batch_norm_layer=BatchNorm)`.

In [ ]:
class BatchNorm(nn.Module):
    """
    A hand-written batch normalization layer. It can be used in place of `nn.BatchNorm1d`.

    :param num_features: int
        The number of nodes (or feature maps) coming out of the layer. We keep one mean and variance for each.
    :param dims: tuple of ints (default (0,))
        The dimensions to calculate the batch mean and variance over. For the (batch, nodes) tensors of a fully
        connected layer that is (0,). For convolutional layers with (batch, channels, height, width) tensors
        it would be (0, 2, 3), giving one mean and variance per feature map.
    :param eps: float
        A small constant, used to ensure we don't divide by zero. This is the value TensorFlow used by default.
    :param momentum: float
        How quickly the population estimates follow the batch statistics. This is `1 - decay`, so 0.01 means
        decay=0.99, the value `tf.layers.batch_normalization` used by default.
    """

    def __init__(self, num_features, dims=(0,), eps=1e-3, momentum=0.01):
        super().__init__()
        self.dims, self.eps, self.momentum = dims, eps, momentum

        # Batch normalization adds additional trainable variables:
        # gamma (for scaling) and beta (for shifting).
        # They start at 1 and 0, so they begin with no effect: y = gamma * x_hat + beta
        self.gamma = nn.Parameter(torch.ones(num_features))
        self.beta = nn.Parameter(torch.zeros(num_features))

        # These buffers store the mean and variance for this layer over the entire training set,
        # which we assume represents the general population distribution.
        # Buffers are saved with the model but are NOT trained by back propagation. Instead,
        # we update them ourselves in `forward`.
        self.register_buffer('pop_mean', torch.zeros(num_features))
        self.register_buffer('pop_variance', torch.ones(num_features))

    def forward(self, x):
        # Shape that lets a per-feature vector broadcast against x, e.g. (1, C) or (1, C, 1, 1)
        shape = [1, -1] + [1] * (x.dim() - 2)

        if self.training:
            # Calculate the mean and variance of the data coming out of this layer's linear-combination step.
            batch_mean = x.mean(dim=self.dims)
            batch_variance = x.var(dim=self.dims, correction=0)

            # Calculate a moving average of the training data's mean and variance while training.
            # These will be used during inference.
            with torch.no_grad():
                self.pop_mean.mul_(1 - self.momentum).add_(self.momentum * batch_mean)
                self.pop_variance.mul_(1 - self.momentum).add_(self.momentum * batch_variance)
            mean, variance = batch_mean, batch_variance
        else:
            # During inference, use the our estimated population mean and variance to normalize the layer
            mean, variance = self.pop_mean, self.pop_variance

        # Normalize, then scale and shift: x_hat = (x - mean) / sqrt(variance + eps); y = gamma * x_hat + beta
        x_hat = (x - mean.view(shape)) / torch.sqrt(variance.view(shape) + self.eps)
        return self.gamma.view(shape) * x_hat + self.beta.view(shape)

This class is longer than just using `nn.BatchNorm1d`, but once again has extensive comments to help you understand it. Here are some important points:

1. It explicitly creates variables to store gamma, beta, and the population mean and variance. These were all handled for us when we used `nn.BatchNorm1d`.
2. It initializes gamma to one and beta to zero, so they start out having no effect in this calculation: $y_i \leftarrow \gamma \hat{x_i} + \beta$. However, during training the network learns the best values for these variables using back propagation, just like networks normally do with weights.
3. Unlike gamma and beta, the population mean and variance are *buffers* (`register_buffer`) rather than parameters. That tells PyTorch not to modify them during back propagation. Instead, the lines that call `mul_` and `add_` update these buffers directly, in place, every time we run a batch in training mode.
4. We wrap those updates in `torch.no_grad()` so PyTorch doesn't try to track them as part of the gradient calculation.
5. We need to handle training and inference differently. PyTorch modules have a `self.training` flag that `network.train()` and `network.eval()` set for every layer inside, so we simply check it with an ordinary `if`.
6. We use `x.mean` and `x.var` to calculate the batch mean and variance. (`correction=0` divides by the number of values, like the batch normalization equations do.)

**2)** The `train` method in `NeuralNet` works fine with this new layer without any changes. It calls `self.network.train()` before every training step, so the population statistics are updated, and `self.network.train(False)` (inference mode) when it measures accuracy, so they are used.

**3)** If you want to see exactly what each line does, you can write the normalization step with plain operations. The two lines in `forward` that compute `x_hat` and the result are the following two equations written out directly. The first line calculates the following equation, with `x` representing $x_i$ and `x_hat` representing $\hat{x_i}$:

$$
\hat{x_i} \leftarrow \frac{x_i - \mu_B}{\sqrt{\sigma_{B}^{2} + \epsilon}}
$$

And the second line is a direct translation of the following equation:

$$
y_i \leftarrow \gamma \hat{x_i} + \beta
$$

We still use `x.mean` and `x.var` to implement the other two equations from earlier – the ones that calculate the batch mean and variance used in the normalization step. If you really wanted to do everything from scratch, you could replace those, too, but we'll leave that to you.

## Why the difference between training and inference?

With `nn.BatchNorm1d` (or our own `BatchNorm`) we tell the layer whether or not the network is training by switching the whole network between modes, like so:
```python
network.train()   # training mode: normalize with the batch's mean and variance
network.eval()    # inference mode: normalize with the estimated population mean and variance
```
If you looked at the [low level implementation](#implementation_2), you probably noticed that we need to do slightly different things during training and inference. But why is that?

First, let's look at what happens when we don't. The following function is similar to `train_and_test` from earlier, but this time we are only testing one network and instead of plotting its accuracy, we perform 200 predictions on test inputs, 1 input at at time. We can use the `test_training_accuracy` parameter to test the network in training or inference modes (the equivalent of calling `network.train()` or `network.eval()` before predicting).

**Note:** we use our own `BatchNorm` layer for this demonstration. PyTorch's built-in `nn.BatchNorm1d` refuses to run in training mode on a batch of one example and raises an error, which is a helpful guard rail but would stop us from seeing what actually goes wrong.

In [ ]:
def batch_norm_test(test_training_accuracy):
    """
    :param test_training_accuracy: bool
        If True, perform inference with batch normalization using batch mean and variance;
        if False, perform inference with batch normalization using estimated population mean and variance.
    """

    weights = [np.random.normal(size=(784,100), scale=0.05).astype(np.float32),
               np.random.normal(size=(100,100), scale=0.05).astype(np.float32),
               np.random.normal(size=(100,100), scale=0.05).astype(np.float32),
               np.random.normal(size=(100,10), scale=0.05).astype(np.float32)
              ]

    # Train the model
    bn = NeuralNet(weights, torch.relu, True, batch_norm_layer=BatchNorm)

    # First train the network
    bn.train(0.01, 2000, 2000)

    bn.test(test_training_accuracy=test_training_accuracy, include_individual_predictions=True)

In the following cell, we pass `True` for `test_training_accuracy`, which performs the same batch normalization that we normally perform **during training**.

In [ ]:
batch_norm_test(True)

As you can see, the network guessed the same value every time! But why? Because during training, a network with batch normalization adjusts the values at each layer based on the mean and variance **of that batch**. The "batches" we are using for these predictions have a single input each time, so their values _are_ the means, and their variances will always be 0. That means the network will normalize the values at any layer to zero. (Review the equations from before to see why a value that is equal to the mean would always normalize to zero.) So we end up with the same result for every input we give the network, because its the value the network produces when it applies its learned weights to zeros at every layer.

**Note:** If you re-run that cell, you might get a different value from what we showed. That's because the specific weights the network learns will be different every time. But whatever value it is, it should be the same for all 200 predictions.

To overcome this problem, the network does not just normalize the batch at each layer. It also maintains an estimate of the mean and variance for the entire population. So when we perform inference, instead of letting it "normalize" all the values using their own means and variance, it uses the estimates of the population mean and variance that it calculated while training.

So in the following example, we pass `False` for `test_training_accuracy`, which tells the network that we it want to perform inference with the population statistics it calculates during training.

In [ ]:
batch_norm_test(False)

As you can see, now that we're using the estimated population mean and variance, we get a 97% accuracy. That means it guessed correctly on 194 of the 200 samples – not too bad for something that trained in under 4 seconds. :)

# Considerations for other network types

This notebook demonstrates batch normalization in a standard neural network with fully connected layers. You can also use batch normalization in other types of networks, but there are some special considerations.

### ConvNets

Convolution layers consist of multiple feature maps. (Remember, the depth of a convolutional layer refers to its number of feature maps.) And the weights for each feature map are shared across all the inputs that feed into the layer. Because of these differences, batch normalizaing convolutional layers requires batch/population mean and variance per feature map rather than per node in the layer.

For convolutional layers use `nn.BatchNorm2d`. PyTorch tensors are laid out as (batch, channels, height, width), so the "feature" it normalizes is dimension 1, the channels.

In our low-level implementation, we used the following line to calculate the batch mean and variance:
```python
batch_mean = x.mean(dim=(0,))
```
If we were dealing with a convolutional layer, we would calculate the mean and variance with a line like this instead:
```python
batch_mean = x.mean(dim=(0, 2, 3))
```
The `dim` argument, `(0, 2, 3)`, tells PyTorch to calculate the batch mean and variance over each feature map. (The three dimensions are the batch, height, and width.) That leaves one mean/variance pair per feature map. That is exactly what the `dims` parameter of our `BatchNorm` class is for: `BatchNorm(num_channels, dims=(0, 2, 3))`.

### RNNs

Batch normalization can work with recurrent neural networks, too, as shown in the 2016 paper [Recurrent Batch Normalization](https://arxiv.org/abs/1603.09025). It's a bit more work to implement, but basically involves calculating the means and variances per time step instead of per layer. Layer normalization (`nn.LayerNorm`), which normalizes over the features of each example instead of over the batch, is a popular alternative for recurrent networks.